# LSTM Random Search Hyperparameter Optimisation

Derived directly from the uploaded source notebook. Random Search tunes the model using validation F1; the held-out test set is untouched.

## 1. Environment Setup

Run this notebook from a clean Google Colab session or locally.


## 2. Package Installation


This cell checks whether the libraries required by the optional LSTM workflow are installed and installs only the missing packages before the remaining audio-processing, analysis, plotting, and modelling steps run.

In [ ]:
# Purpose: Checks whether the libraries required by the optional LSTM workflow are installed and
# installs only the missing packages before the remaining audio-processing, analysis, plotting,
# and modelling steps run.
import importlib.util
import subprocess
import sys

# List the import names and their corresponding installable package names.
required_packages = [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("tensorflow", "tensorflow"),
]

# Install only dependencies that are not already available in the active kernel.
missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already installed.")


## 3. Imports


This cell imports the standard-library and third-party tools used by the optional LSTM workflow and provides a print-based fallback when the richer notebook display function is unavailable.

In [ ]:
# Purpose: Imports the standard-library and third-party tools used by the optional LSTM workflow
# and provides a print-based fallback when the richer notebook display function is unavailable.
import json
import os
import random
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print


## 4. Configuration and Random Seeds


This cell fixes the available random seeds for reproducibility and defines the shared audio, MFCC, class, sampling, and optional LSTM training settings used later in the notebook.

In [ ]:
# Purpose: Fixes the available random seeds for reproducibility and defines the shared audio,
# MFCC, class, sampling, and optional LSTM training settings used later in the notebook.
# Use a fixed seed so sampling, splitting, and model initialisation can be repeated.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

# Configure the fixed audio representation and MFCC analysis window.
SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5.0
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))
# Derive the fixed MFCC time dimension from the audio and framing settings.
EXPECTED_FRAMES = 1 + max(0, int(SAMPLE_RATE * FIXED_DURATION_SECONDS) - N_FFT) // HOP_LENGTH

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

# Keep dataset and training limits together so quick runs are easy to configure.
MAX_FILES_PER_CLASS = None  # Set to a small number, such as 40, for a quick check.
EPOCHS = 30
BATCH_SIZE = 128


## 5. Dataset Paths


This cell defines and calls a portable helper that mounts Google Drive when Colab is available and otherwise continues without failing in a local environment.

In [ ]:
# Purpose: Defines and calls a portable helper that mounts Google Drive when Colab is available
# and otherwise continues without failing in a local environment.
# Mount Google Drive when the notebook is running in Colab.
def mount_drive_if_colab(mount_point="/content/drive"):
    try:
        from google.colab import drive
        drive.mount(mount_point)
    except Exception:
        print("Not running in Colab, or Google Drive is already available.")

mount_drive_if_colab()


This cell defines the synthetic and bona-fide dataset paths, creates the output folders required by the optional LSTM workflow, and prints the active locations for confirmation.

In [ ]:
# Purpose: Defines the synthetic and bona-fide dataset paths, creates the output folders
# required by the optional LSTM workflow, and prints the active locations for confirmation.
# Change these paths to match your Google Drive dataset folders.
# Set the shared root used to locate the two audio classes.
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / "Datasets" / "MLAAD_10pct"
BONA_FIDE_AUDIO_DIR = PROJECT_ROOT / "Datasets" / "M_AILABS_bona_fide_subset"

# Build a model-specific output hierarchy under the project root.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lstm_optional"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"

# Create every output directory before any files are saved.
for directory in [OUTPUT_DIR, FIGURES_DIR, METRICS_DIR, MODELS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Synthetic audio directory:", SYNTHETIC_AUDIO_DIR)
print("Bona-fide audio directory:", BONA_FIDE_AUDIO_DIR)
print("Output directory:", OUTPUT_DIR)


## 6. Dataset Loading / Audio File Scan


This cell defines a recursive scanner that validates a dataset folder and records each supported audio file's path, class label, inferred language, and size in a Pandas table.

In [ ]:
# Purpose: Defines a recursive scanner that validates a dataset folder and records each
# supported audio file's path, class label, inferred language, and size in a Pandas table.
# Convert the supported files under one class directory into manifest rows.
def scan_audio_files(root_dir, label, class_name):
    root_dir = Path(root_dir)
    if not root_dir.exists():
        raise FileNotFoundError(f"Folder not found: {root_dir}")

    rows = []
    for path in sorted(root_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            rows.append({
                "path": str(path),
                "relative_path": str(path.relative_to(root_dir)),
                "label": label,
                "class_name": class_name,
                "language": path.parent.name,
                "file_size_mb": path.stat().st_size / (1024 * 1024),
            })
    return pd.DataFrame(rows)


This cell scans both class directories, optionally limits each class for a quick run, combines and reproducibly shuffles the records into one manifest, and displays a short preview.

In [ ]:
# Purpose: Scans both class directories, optionally limits each class for a quick run, combines
# and reproducibly shuffles the records into one manifest, and displays a short preview.
# Scan the synthetic and bona-fide directories separately with their correct labels.
synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, label=1, class_name=CLASS_NAMES[1])
bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, label=0, class_name=CLASS_NAMES[0])

# Optionally draw a reproducible smaller sample for a quick test run.
if MAX_FILES_PER_CLASS:
    synthetic_manifest = synthetic_manifest.sample(min(MAX_FILES_PER_CLASS, len(synthetic_manifest)), random_state=RANDOM_STATE)
    bona_fide_manifest = bona_fide_manifest.sample(min(MAX_FILES_PER_CLASS, len(bona_fide_manifest)), random_state=RANDOM_STATE)

# Merge both classes into one shuffled manifest for later splitting.
manifest = pd.concat([bona_fide_manifest, synthetic_manifest], ignore_index=True)
manifest = manifest.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print("Total files:", len(manifest))
display(manifest.head())


## 7. Dataset Inspection


This cell summarises the manifest by class and language and displays the overall class counts together with the 20 most frequent class-language combinations.

In [ ]:
# Purpose: Summarises the manifest by class and language and displays the overall class counts
# together with the 20 most frequent class-language combinations.
# Count recordings by class and by class-language combination.
class_summary = manifest.groupby(["label", "class_name"]).size().reset_index(name="files")
language_summary = manifest.groupby(["class_name", "language"]).size().reset_index(name="files")

display(class_summary)
display(language_summary.sort_values("files", ascending=False).head(20))


## 8. Data Quality Checks


This cell measures missing paths, files absent from disk, duplicate paths, and represented classes, then stops the workflow if both labels are not present or duplicate recordings could compromise the experiment.

In [ ]:
# Purpose: Measures missing paths, files absent from disk, duplicate paths, and represented
# classes, then stops the workflow if both labels are not present or duplicate recordings could
# compromise the experiment.
# Calculate manifest checks before any expensive audio processing begins.
quality_checks = pd.DataFrame([
    {"check": "missing_paths", "value": int(manifest["path"].isna().sum())},
    {"check": "missing_files", "value": int((~manifest["path"].map(lambda p: Path(p).exists())).sum())},
    {"check": "duplicate_paths", "value": int(manifest["path"].duplicated().sum())},
    {"check": "classes_present", "value": ", ".join(map(str, sorted(manifest["label"].unique())))},
])

display(quality_checks)

# Stop early if the manifest does not contain both required target classes.
if set(manifest["label"].unique()) != {0, 1}:
    raise ValueError("Both classes are required: 0=bona-fide and 1=synthetic/deepfake.")
if manifest["path"].duplicated().any():
    raise ValueError("Duplicate audio paths were found. Remove duplicates before training.")


## 9. Audio Preprocessing

Audio is loaded as mono, resampled, padded or truncated, and normalised.


## 10. MFCC Sequence Extraction


This cell standardises each recording, extracts a fixed-length MFCC matrix, and transposes it into a sequence of time frames with MFCC coefficients that can be processed by the LSTM.

In [ ]:
# Purpose: Standardises each recording, extracts a fixed-length MFCC matrix, and transposes it
# into a sequence of time frames with MFCC coefficients that can be processed by the LSTM.
# Resample, pad or truncate, and peak-normalise every waveform consistently.
def load_audio_fixed(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    target_length = int(SAMPLE_RATE * FIXED_DURATION_SECONDS)

    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    return audio.astype(np.float32)


# Transform the standardised waveform into the MFCC representation required downstream.
def extract_mfcc_sequence(path):
    audio = load_audio_fixed(path)
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        center=False,
    ).astype(np.float32)

    if mfcc.shape[1] < EXPECTED_FRAMES:
        mfcc = np.pad(mfcc, ((0, 0), (0, EXPECTED_FRAMES - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :EXPECTED_FRAMES]

    return mfcc.T


## 11. Label Preparation


This cell states the binary label mapping and displays the number of bona-fide and synthetic recordings so the target variable can be checked before splitting and modelling.

In [ ]:
# Purpose: States the binary label mapping and displays the number of bona-fide and synthetic
# recordings so the target variable can be checked before splitting and modelling.
print("Label meaning: 0 = bona-fide, 1 = synthetic/deepfake")
display(manifest["label"].value_counts().sort_index().rename(index=CLASS_NAMES).to_frame("files"))


## 12. Train / Validation / Test Split


This cell creates reproducible stratified training, validation, and test partitions in a 70:15:15 ratio, resets their row indexes, and displays the size and class balance of each split.

In [ ]:
# Purpose: Creates reproducible stratified training, validation, and test partitions in a
# 70:15:15 ratio, resets their row indexes, and displays the size and class balance of each
# split.
# Reserve 30 percent of the manifest before dividing it equally into validation and test data.
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=manifest["label"],
)

# Split the held-out portion equally while preserving the class distribution.
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "bona_fide": [int((train_df["label"] == 0).sum()), int((validation_df["label"] == 0).sum()), int((test_df["label"] == 0).sum())],
    "synthetic": [int((train_df["label"] == 1).sum()), int((validation_df["label"] == 1).sum()), int((test_df["label"] == 1).sum())],
})
display(split_summary)


## 13. LSTM Sequence Feature Preparation


This cell standardises each recording, extracts a fixed-length MFCC matrix, and transposes it into a sequence of time frames with MFCC coefficients that can be processed by the LSTM.

In [ ]:
def build_sequence_table(df, split_name):
    sequences = []
    labels = []
    kept_rows = []

    for _, row in df.iterrows():
        try:
            sequences.append(extract_mfcc_sequence(row["path"]))
            labels.append(int(row["label"]))
            kept_rows.append(row)
        except Exception as exc:
            print(f"Skipping {row['path']}: {exc}")

    if not sequences: # Handle the case where no sequences were successfully extracted
        print(f"Warning: No sequences extracted for {split_name}. Returning empty arrays.")
        # Assuming EXPECTED_FRAMES and N_MFCC are defined globally or passed as arguments
        return np.array([]).reshape(0, EXPECTED_FRAMES, N_MFCC), np.array([], dtype=np.int64), pd.DataFrame()

    X = np.stack(sequences).astype(np.float32)
    y = np.array(labels, dtype=np.int64)
    meta = pd.DataFrame(kept_rows).reset_index(drop=True)
    print(f"{split_name}: {X.shape}")
    return X, y, meta

X_train, y_train, train_meta = build_sequence_table(train_df, "train")
X_validation, y_validation, validation_meta = build_sequence_table(validation_df, "validation")


In [ ]:
# Test the first file from the validation set that failed
test_path = validation_df.iloc[0]['path']

print(f"Testing path: {test_path}")
print(f"Exists according to os.path: {os.path.exists(test_path)}")

if os.path.exists(test_path):
    print(f"Is a standard file: {os.path.isfile(test_path)}")
    print(f"File size: {os.path.getsize(test_path)} bytes")

    # Try to open and read the first few bytes to check if it's readable
    try:
        with open(test_path, 'rb') as f:
            header = f.read(12)
            print(f"File header (first 12 bytes): {header}")
            print("File can be opened successfully! The issue might be intermittent Drive timeouts.")
    except Exception as e:
        print(f"Error opening file directly: {e}")
else:
    print("\nThe file truly does not exist on disk right now.")
    print("Try remounting Google Drive: click the 'Folder' icon on the left, then the 'Drive' icon with a slash to unmount, and run the drive mounting cell again.")


## 14. Train-only MFCC Sequence Standardisation


This cell calculates MFCC-sequence normalisation statistics from the training split only and applies them to both the training and validation tensors before reporting the LSTM input shape.

In [ ]:
# Purpose: Calculates MFCC-sequence normalisation statistics from the training split only and
# applies them to both the training and validation tensors before reporting the LSTM input
# shape.
# Estimate normalisation statistics from the training tensor only.
mfcc_mean = X_train.mean(axis=(0, 1), keepdims=True)
mfcc_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6

X_train_scaled = (X_train - mfcc_mean) / mfcc_std
X_validation_scaled = (X_validation - mfcc_mean) / mfcc_std

print("MFCC sequence standardisation fitted on training data only.")
print("LSTM input shape:", X_train_scaled.shape[1:])


## 15. LSTM Model Definition


This cell constructs and summarises an example sequence classifier containing an LSTM layer, dropout regularisation, and a sigmoid output for binary classification.

In [ ]:
# Purpose: Constructs and summarises an example sequence classifier containing an LSTM layer,
# dropout regularisation, and a sigmoid output for binary classification.
# Construct a representative estimator so its architecture or configuration can be inspected.
example_model = Sequential()
example_model.add(Input(shape=X_train_scaled.shape[1:]))
example_model.add(LSTM(64))
example_model.add(Dropout(0.30))
example_model.add(Dense(1, activation="sigmoid"))

example_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
example_model.summary()


## 16. Training


This cell documents that model fitting happens in the following validation experiment and that early stopping uses validation loss while restoring the best recorded epoch.

In [ ]:
# Purpose: Documents that model fitting happens in the following validation experiment and that
# early stopping uses validation loss while restoring the best recorded epoch.
# The model is trained in the validation experiment loop below.
# Early stopping watches validation loss and restores the best validation epoch.


## 17. Random Search Hyperparameter Optimisation

This cell trains the candidate LSTM configurations with early stopping and best-checkpoint saving, measures their validation performance, ranks them by F1 score, and reloads the selected model.

In [ ]:
# Purpose: Tune the uploaded LSTM using reproducible Random Search and validation F1.
# The held-out test split is never used during this search.
OPTIMISATION_DIR = OUTPUT_DIR / "optimisation"
OPTIMISATION_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEARCH_TRIALS = 12
SEARCH_SPACE = {'units': [32, 64, 128], 'dropout': [0.2, 0.3, 0.4, 0.5], 'learning_rate': [0.0001, 0.0003, 0.001, 0.003], 'batch_size': [32, 64, 128]}

def build_model(config):
    """Build the source-faithful single-layer LSTM from one hyperparameter configuration."""
    model = Sequential()
    model.add(Input(shape=X_train_scaled.shape[1:]))
    model.add(LSTM(int(config["units"])))
    model.add(Dropout(float(config["dropout"])))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=float(config["learning_rate"]))
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model

def all_configurations(space):
    """Expand the discrete search space into unique configurations for sampling."""
    keys = list(space)
    configs = [{}]
    for key in keys:
        configs = [{**cfg, key: value} for cfg in configs for value in space[key]]
    return configs

rng = random.Random(RANDOM_STATE)
all_configs = all_configurations(SEARCH_SPACE)
trial_configs = rng.sample(all_configs, k=min(RANDOM_SEARCH_TRIALS, len(all_configs)))
results = []

for trial_number, config in enumerate(trial_configs, start=1):
    print(f"\nRandom Search trial {trial_number}/{len(trial_configs)}: {config}")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(config)
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
    history = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_validation_scaled, y_validation),
        epochs=EPOCHS,
        batch_size=int(config["batch_size"]),
        callbacks=callbacks,
        verbose=2,
    )
    probability = model.predict(X_validation_scaled, batch_size=int(config["batch_size"])).ravel()
    pred = (probability >= 0.5).astype(int)
    results.append({
        "trial": trial_number,
        "config": config,
        "accuracy": float(accuracy_score(y_validation, pred)),
        "precision": float(precision_score(y_validation, pred, zero_division=0)),
        "recall": float(recall_score(y_validation, pred, zero_division=0)),
        "f1": float(f1_score(y_validation, pred, zero_division=0)),
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
    })

results_df = pd.DataFrame([{"trial": r["trial"], **r["config"], "accuracy": r["accuracy"], "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "best_val_loss": r["best_val_loss"], "best_epoch": r["best_epoch"]} for r in results]).sort_values("f1", ascending=False)
display(results_df)
best = max(results, key=lambda r: (r["f1"], -r["best_val_loss"]))
best_payload = {"method": "random_search", "model": 'Optional LSTM', "validation_f1": best["f1"], "validation_accuracy": best["accuracy"], "best_val_loss": best["best_val_loss"], "config": best["config"]}

results_df.to_csv(OPTIMISATION_DIR / "random_search_results.csv", index=False)
with open(OPTIMISATION_DIR / "random_search_best.json", "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)
print("Best Random Search configuration:", best_payload)
print("Saved:", OPTIMISATION_DIR / "random_search_best.json")


## Test Set Deliberately Unused

Random Search performs model selection using validation data only. Do not evaluate the test set in this notebook.

## 21. Reproducibility Checks


This cell displays a compact reproducibility record for the optional LSTM workflow, including the stratified split, train-only preprocessing, test-set isolation, label mapping, dependency design, and random seed.

In [ ]:
# Purpose: Displays a compact reproducibility record for the optional LSTM workflow, including
# the stratified split, train-only preprocessing, test-set isolation, label mapping, dependency
# design, and random seed.
# Collect the main reproducibility and leakage-prevention properties for display.
checks = {
    "self_contained": True,
    "external_helper_file_required": False,
    "custom_helper_imports": False,
    "optional_model": True,
    "core_model": False,
    "split": "70/15/15 stratified by class label",
    "standardisation_fitted_on": "training split only",
    "test_used_for_model_selection": False,
    "label_mapping": CLASS_NAMES,
    "random_state": RANDOM_STATE,
}

display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
